In [1]:
import os
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("../data/raw/2019-Oct.csv")

In [5]:
rows_before = len(df)

print("Rows before cleaning:", rows_before)
print("Missing category_code:", df["category_code"].isna().sum())
print("Missing brand:", df["brand"].isna().sum())
print("Missing sessions:", df["user_session"].isna().sum())
print("Exact duplicates:", df.duplicated().sum())
print("Zero prices:", (df["price"] == 0).sum())

Rows before cleaning: 42448764
Missing category_code: 13515609
Missing brand: 6117080
Missing sessions: 2
Exact duplicates: 30220
Zero prices: 68673


In [7]:
event_counts_before = df["event_type"].value_counts().copy()

event_counts_before

event_type
view        40779399
cart          926516
purchase      742849
Name: count, dtype: int64

In [13]:
df["category_code"] = df["category_code"].fillna("uncategorized")

In [11]:
df["category_code"].isna().sum()

0

Missing category_code values could not be reconstructed from the available data. Rather than assigning unsupported product categories or removing valid customer events, missing values were retained as a separate uncategorized group.

In [19]:
known_product_brands = (
    df.loc[df["brand"].notna(), ["product_id", "brand"]]
      .drop_duplicates()
)

In [21]:
brand_count_per_product = (
    known_product_brands
    .groupby("product_id")["brand"]
    .nunique()
)

In [23]:
single_brand_products = brand_count_per_product[
    brand_count_per_product == 1
].index

In [25]:
brand_lookup = (
    known_product_brands[
        known_product_brands["product_id"].isin(single_brand_products)
    ]
    .drop_duplicates("product_id")
    .set_index("product_id")["brand"]
)

In [27]:
missing_brand_mask = df["brand"].isna()

df.loc[missing_brand_mask, "brand"] = (
    df.loc[missing_brand_mask, "product_id"]
      .map(brand_lookup)
)

In [29]:
remaining_missing_brand = df["brand"].isna().sum()

print("Remaining missing brand:", remaining_missing_brand)

Remaining missing brand: 5945324


In [31]:
original_missing_brand = int(len(df) * 0.1441)

print(
    "Approximately recovered:",
    original_missing_brand - remaining_missing_brand
)

Approximately recovered: 171542


In [33]:
df["brand"] = df["brand"].fillna("unknown")

In [35]:
df["brand"].isna().sum()

0

Missing brands were first recovered only where a product had one consistent known brand elsewhere in the dataset. Remaining missing values were labelled unknown rather than inferred or removed.

In [38]:
multi_brand_ids = brand_count_per_product[
    brand_count_per_product > 1
].index

multi_brand_review = (
    df[df["product_id"].isin(multi_brand_ids)]
    [["product_id", "brand", "category_id", "category_code"]]
    .drop_duplicates()
    .sort_values("product_id")
)

multi_brand_review.head(20)

,product_id,brand,category_id,category_code
26144151,1306881,lenovo,2053013558920217191,computers.notebook
322358,1306881,dell,2053013558920217191,computers.notebook
39188202,2800241,indesit,2053013563835941749,appliances.kitchen.refrigerators
407768,2800241,leadbros,2053013563835941749,appliances.kitchen.refrigerators
5523,4300224,timberk,2053013552385491165,uncategorized
34310842,4300224,ibaby,2053013552385491165,uncategorized
22089104,4502735,unknown,2053013563877884791,appliances.kitchen.hob
22334031,4502735,dauscher,2053013563877884791,appliances.kitchen.hob
22993239,4502735,maunfeld,2053013563877884791,appliances.kitchen.hob
22239558,4502742,indesit,2053013563877884791,appliances.kitchen.hob


In [42]:
multi_brand_review.to_csv(
    "../data/processed/multi_brand_review.csv",
    index=False
)

In [44]:
duplicate_review = df[
    df.duplicated(keep=False)
].sort_values(
    ["user_session", "event_time", "product_id"]
)

duplicate_review.to_csv(
    "../data/processed/duplicate_review.csv",
    index=False
)

In [46]:
zero_price_review = df[df["price"] == 0]

zero_price_review["event_type"].value_counts()

event_type
view    68523
cart      150
Name: count, dtype: int64

In [48]:
zero_price_review[
    ["event_type", "product_id", "category_code", "brand", "price"]
].head(20)

,event_type,product_id,category_code,brand,price
2259,view,53000001,uncategorized,nike,0.0
2312,view,53000001,uncategorized,nike,0.0
2600,view,7000684,kids.carriage,peg-perego,0.0
3367,view,4100157,uncategorized,nintendo,0.0
4941,view,23301316,uncategorized,unknown,0.0
5007,view,13105134,uncategorized,unknown,0.0
5843,view,31200824,uncategorized,unknown,0.0
5947,view,23301316,uncategorized,unknown,0.0
5951,view,31200824,uncategorized,unknown,0.0
9514,view,32403740,uncategorized,unknown,0.0


In [50]:
zero_price_review.to_csv(
    "../data/processed/zero_price_review.csv",
    index=False
)

In [52]:
df[df["user_session"].isna()]

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
7157243,2019-10-06 14:26:10 UTC,cart,1801723,2053013554415534427,electronics.video.tv,tcl,135.65,557388939,NaN
33940774,2019-10-25 10:36:14 UTC,cart,1004767,2053013555631882655,electronics.smartphone,samsung,246.52,549825742,NaN


In [54]:
df["event_time"] = pd.to_datetime(
    df["event_time"],
    utc=True
)

In [55]:
df["event_time"].dtype

datetime64[ns, UTC]

In [58]:
df["event_type"] = df["event_type"].astype("category")

In [60]:
df["category_code"] = df["category_code"].astype("category")

In [61]:
df["brand"] = df["brand"].astype("category")

In [64]:
df["product_id"] = pd.to_numeric(
    df["product_id"],
    downcast="unsigned"
)

df["category_id"] = pd.to_numeric(
    df["category_id"],
    downcast="unsigned"
)

df["user_id"] = pd.to_numeric(
    df["user_id"],
    downcast="unsigned"
)

In [66]:
rows_after = len(df)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Difference:", rows_before - rows_after)

Rows before: 42448764
Rows after: 42448764
Difference: 0


In [68]:
event_counts_after = df["event_type"].value_counts()

pd.DataFrame({
    "before": event_counts_before,
    "after": event_counts_after
})

,before,after
event_type,,
view,40779399,40779399
cart,926516,926516
purchase,742849,742849


In [70]:
df.isna().sum()

event_time       0
event_type       0
product_id       0
category_id      0
category_code    0
brand            0
price            0
user_id          0
user_session     2
dtype: int64

In [72]:
print("Zero prices:", (df["price"] == 0).sum())
print("Negative prices:", (df["price"] < 0).sum())

Zero prices: 68673
Negative prices: 0


In [74]:
memory_gb = df.memory_usage(deep=True).sum() / (1024 ** 3)

print(f"Current memory usage: {memory_gb:.2f} GB")

Current memory usage: 5.14 GB


In [76]:
df.to_parquet(
    "../data/processed/2019-Oct-cleaned.parquet",
    index=False
)

In [78]:
df.dtypes


event_time       datetime64[ns, UTC]
event_type                  category
product_id                    uint32
category_id                   uint64
category_code               category
brand                       category
price                        float64
user_id                       uint32
user_session                  object
dtype: object

In [80]:
df.isna().sum()

event_time       0
event_type       0
product_id       0
category_id      0
category_code    0
brand            0
price            0
user_id          0
user_session     2
dtype: int64

In [82]:

df.memory_usage(deep=True).sum() / (1024 ** 3)

5.13969496730715

In [87]:
df["brand"].value_counts().get("unknown", 0)

5945324

# STEP 3 — Data Cleaning

## Objective

The purpose of Day 3 was to clean the major data-quality issues identified during the Day 2 audit while avoiding unsupported assumptions or unnecessary data deletion.

The cleaning strategy focused on preserving valid behavioural events wherever possible.

## 1. Missing Category Codes

Approximately **31.84%** of `category_code` values were missing.

The Day 2 investigation showed that missing category codes could not be reconstructed from other records using `category_id`.

### Cleaning Decision

Missing category codes were therefore labelled:

`uncategorized`

rather than assigning an unsupported category or deleting the affected events.

After cleaning:

- Missing `category_code`: **0**

## 2. Missing Brand Information

Approximately **14.41%** of records originally contained missing brand information.

Where a product had exactly one consistent known brand elsewhere in the dataset, that known product-brand relationship was used to recover missing brand information.

Remaining missing brand values that could not be reliably reconstructed were labelled:

`unknown`

### Cleaning Decision

Brand values were only reconstructed where supporting evidence existed from the same `product_id`.

No brand was guessed or manually assigned.

After cleaning:

- Missing `brand`: **0**

## 3. Products Associated With Multiple Brands

The audit identified **137 products** associated with more than one non-null brand.

Inspection showed that some differences represented genuinely different brands rather than simple spelling or capitalisation differences.

### Cleaning Decision

These records were retained unchanged.

The correct brand could not be reliably determined from the available data, so forcing one brand assignment could introduce inaccurate information.

The issue was documented as a source-data limitation.

## 4. Duplicate-Looking Events

The audit identified:

- **30,220 additional exact duplicate rows**
- approximately **0.0712%** of the dataset
- **48,319 rows** involved in duplicate groups when all copies were included

Most duplicate-looking records were cart events.

### Final Brand Cleaning Result

After attempting to recover missing brand values using products that had exactly one consistent known brand elsewhere in the dataset:

- **5,945,324 records** still had no reliable brand information.
- These records were labelled `unknown`.
- No unsupported brand values were inferred or assigned.

This preserves the events for behavioural analysis while clearly separating records where brand information is unavailable.